#Data Processing

In [2]:
!apt-get install openjdk-8-jdk-headless -qq
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewsCleaning") \
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED") \
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .config("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS") \
    .getOrCreate()

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np
import pandas as pd
import re

# Import Spark functions
from pyspark.sql.functions import *

##Review Data Processing

In [5]:
# Review dataset columns
df_sample = pd.read_json(
    "/content/drive/MyDrive/Kpdl1/reviews_Electronics.jsonl",
    lines=True,
    nrows=2000
)

df_sample.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')

###Convert reviews data to Parquet and select required columns



In [6]:
reviews_in = "/content/drive/MyDrive/Kpdl1/reviews_Electronics.jsonl"
reviews_out = "/content/drive/MyDrive/Kpdl1/reviews_Electronics.parquet"

reviews = spark.read.json(reviews_in)

# Select required columns
reviews_cols = [
    "asin",
    "parent_asin",
    "rating",
    "title",
    "text",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

reviews = reviews.select(*reviews_cols)

# Repartition dataset to enable parallel writing
reviews = reviews.repartition(8)

# Save as Parquet
reviews.write.mode("overwrite").parquet(reviews_out)

###Load reviewdata

In [6]:
reviews = spark.read.parquet("/content/drive/MyDrive/Kpdl1/reviews_Electronics.parquet")

print("Number of rows:", reviews.count())
print("Columns:", reviews.columns)

reviews.show(5)

Number of rows: 43886944
Columns: ['asin', 'parent_asin', 'rating', 'title', 'text', 'timestamp', 'helpful_vote', 'verified_purchase']
+----------+-----------+------+--------------------+--------------------+-------------+------------+-----------------+
|      asin|parent_asin|rating|               title|                text|    timestamp|helpful_vote|verified_purchase|
+----------+-----------+------+--------------------+--------------------+-------------+------------+-----------------+
|B0035PBHX6| B0035PBHX6|   4.0|Really good the p...|This product is g...|1367706507000|           0|             true|
|B00F0XQWUO| B00F0XQWUO|   1.0|           Forget it|Read the plan fir...|1485662298000|           0|             true|
|B078B3F3K6| B0789TZCRQ|   5.0|      Great product!|Great size, great...|1540454603096|           0|             true|
|B006GCYAOI| B0C5LRK29P|   5.0|Got me working ag...|When I first move...|1530455710590|           0|            false|
|B085NPCQLT| B0BMXMNG7G|   5.0|g

In [7]:
# Check data types of each column
reviews.printSchema()

root
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [9]:
# Check missing values for each column
missing = reviews.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in reviews.columns
])

missing.show()

+----+-----------+------+-----+----+---------+------------+-----------------+
|asin|parent_asin|rating|title|text|timestamp|helpful_vote|verified_purchase|
+----+-----------+------+-----+----+---------+------------+-----------------+
|   0|          0|     0|    0|   0|        0|           0|                0|
+----+-----------+------+-----+----+---------+------------+-----------------+



In [8]:
total_rows = reviews.count()

empty_parent_asin = reviews.filter(trim(col("parent_asin")) == "").count() / total_rows
empty_title = reviews.filter(trim(col("title")) == "").count() / total_rows
empty_text = reviews.filter(trim(col("text")) == "").count() / total_rows

{
    "empty_parent_asin_ratio": float(empty_parent_asin),
    "empty_title_ratio": float(empty_title),
    "empty_text_ratio": float(empty_text)
}

{'empty_parent_asin_ratio': 0.0,
 'empty_title_ratio': 2.2785819855672794e-08,
 'empty_text_ratio': 0.0009385934914948737}

In [9]:
# Check number of reviews before removal
before_count = reviews.count()

# Remove reviews with empty text
reviews = reviews.filter(trim(col("text")) != "")

# Check number of reviews after removal
after_count = reviews.count()

print("Reviews before:", before_count)
print("Reviews after:", after_count)
print("Removed:", before_count - after_count)

Reviews before: 43886944
Reviews after: 43845752
Removed: 41192


In [10]:
# Create review length
reviews = reviews.withColumn(
    "text_length",
    length(trim(col("text")))
)

# Statistics
reviews.select("text_length").describe().show()

+-------+------------------+
|summary|       text_length|
+-------+------------------+
|  count|          43845752|
|   mean|241.55279373472715|
| stddev| 419.5295122985489|
|    min|                 1|
|    max|             35208|
+-------+------------------+



###Remove reviews with abnormal length









In [11]:
# Remove only extremely long reviews
stats = reviews.select(
    count("*").alias("before"),
    sum(when(col("text_length") > 5000, 1).otherwise(0)).alias("removed")
)

result = stats.collect()[0]

before = result["before"]
removed = result["removed"]

reviews = reviews.filter(col("text_length") <= 5000)

after = before - removed

print("Reviews before:", before)
print("Reviews after:", after)
print("Removed:", removed)

Reviews before: 43845752
Reviews after: 43811736
Removed: 34016


In [12]:
# Drop text_length column
reviews = reviews.drop("text_length")

In [13]:
# Rating distribution and filter valid ratings (1–5)
stats = reviews.select(
    count("*").alias("before"),
    sum(when(~col("rating").between(1,5),1).otherwise(0)).alias("removed")
).collect()[0]

before = stats["before"]
removed = stats["removed"]
after = before - removed

print("Before filter:", before)
print("After filter:", after)
print("Removed:", removed)

reviews = reviews.filter(col("rating").between(1,5))

Before filter: 43811736
After filter: 43811734
Removed: 2


In [14]:
# Remove duplicate
before = reviews.count()

reviews = reviews.dropDuplicates([
    "asin",
    "parent_asin",
    "rating",
    "title",
    "text",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
])

after = reviews.count()

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 43811734
After: 43333151
Removed: 478583


In [15]:
# Normalize title and text
reviews = reviews.withColumn(
    "title",
    trim(
        regexp_replace(
            regexp_replace(lower(coalesce(col("title"), lit(""))), r"[^a-z0-9'\s]", " "),
            r"\s+",
            " "
        )
    )
).withColumn(
    "text",
    trim(
        regexp_replace(
            regexp_replace(lower(col("text")), r"[^a-z0-9'\s]", " "),
            r"\s+",
            " "
        )
    )
)


In [16]:
#Convert timestap
reviews = reviews.withColumn(
    "timestamp",
    from_unixtime(col("timestamp")/1000).cast("timestamp")
)

###Save file

In [ ]:
reviews = reviews.repartition(60)

# Save parquet
save_path = "/content/drive/MyDrive/Kpdl1/clean_reviews.parquet"
reviews.write.mode("overwrite").parquet(save_path)